# 3A - Modeling su Dataset MACRO-ONLY

## 3.1 - Import e Caricamento dati

In [83]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score, classification_report
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Caricamento del dataset
df_macro_lagged = pd.read_csv('./data/df_macro_long_lagged.csv', index_col=0 )

print(f"✓ Dataset preprocessato caricato: {df_macro_lagged.shape[0]} righe e {df_macro_lagged.shape[1]} colonne")
print(f"✓ Colonne disponibili: {df_macro_lagged.columns.tolist()}")
print(f"NaN values per colonna:\n{df_macro_lagged.isna().sum()}")

✓ Dataset preprocessato caricato: 9373 righe e 66 colonne
✓ Colonne disponibili: ['ECB_Deposit_Rate', 'ECB_MRO_Rate', 'FED_Rate', 'VIX', 'ESI_Index', 'Euribor_3M', 'Euribor_1Y', 'HICP_Euroarea', 'interbank_stress_spread', 'short_yield_curve_slope', 'bce_fed_spread', 'ECB_Deposit_Rate_lag_7d', 'ECB_Deposit_Rate_lag_21d', 'ECB_Deposit_Rate_lag_63d', 'ECB_Deposit_Rate_lag_126d', 'ECB_Deposit_Rate_lag_252d', 'ECB_MRO_Rate_lag_7d', 'ECB_MRO_Rate_lag_21d', 'ECB_MRO_Rate_lag_63d', 'ECB_MRO_Rate_lag_126d', 'ECB_MRO_Rate_lag_252d', 'FED_Rate_lag_7d', 'FED_Rate_lag_21d', 'FED_Rate_lag_63d', 'FED_Rate_lag_126d', 'FED_Rate_lag_252d', 'VIX_lag_7d', 'VIX_lag_21d', 'VIX_lag_63d', 'VIX_lag_126d', 'VIX_lag_252d', 'ESI_Index_lag_7d', 'ESI_Index_lag_21d', 'ESI_Index_lag_63d', 'ESI_Index_lag_126d', 'ESI_Index_lag_252d', 'Euribor_3M_lag_7d', 'Euribor_3M_lag_21d', 'Euribor_3M_lag_63d', 'Euribor_3M_lag_126d', 'Euribor_3M_lag_252d', 'Euribor_1Y_lag_7d', 'Euribor_1Y_lag_21d', 'Euribor_1Y_lag_63d', 'Euribor_1Y_

## 3.2 - Target Definition

Decidiamo di considerate come forecast horizon 3 mesi. Un orizionte minore catturerebbe solo il rumore, i tassi di interesse non variano così rapidamente, proviamo a catturare i cambiamenti in un tempo che include almeno due riuonioni della bce (avvengonono ogni 6 settimane) e catturiamo almeno due rilasci del dati sull'inflazione (rilasciati con cadenza mensile)

In [84]:
FORECAST_HORIZON = 90 

# Definiamo target variable
df_macro_lagged['Euribor_3M_target'] = df_macro_lagged['Euribor_3M'].shift(-FORECAST_HORIZON)

#Scartiamo le righe con valori NaN nella colonna target
df_macro_lagged = df_macro_lagged.dropna(subset=['Euribor_3M_target']).copy()

df_macro_lagged['target'] = (df_macro_lagged['Euribor_3M_target'] > df_macro_lagged['Euribor_3M']).astype(int) 

print(f"\nTarget distribution:")
print(df_macro_lagged['target'].value_counts())
print(f"Classe imbalance ratio: {(df_macro_lagged['target'] == 1).sum() / (df_macro_lagged['target'] == 0).sum():.3f}")



Target distribution:
target
0    5122
1    4161
Name: count, dtype: int64
Classe imbalance ratio: 0.812


Siamo contenti con circa una perfetta distribuzione     

## 3.3 - Modello Naive: Persistenza statica 

Valutiamo innanzi tutto la persistenza statica, ovvero quella in cui prevediamo che tra 90 giorni ci sia lo stesso tasso. Considerando che stiamo facendo classificazione direzionale (1=sale, 0=non sale), la nostra colonna target sarà composta da tutti 0.

In [85]:
# Isoliamo X and y
y_true = df_macro_lagged['target']

y_naive_static = np.zeros_like(y_true)  # Prevediamo sempre la classe 0 (Euribor_3M non aumenterà)
print("--- Baseline 1: Static Persistence ---")
print(f"\nClassification Report:\n{classification_report(y_true, y_naive_static, zero_division=0)}") # zero_division=0 per evitare warning in caso di classi non predette

--- Baseline 1: Static Persistence ---

Classification Report:
              precision    recall  f1-score   support

           0       0.55      1.00      0.71      5122
           1       0.00      0.00      0.00      4161

    accuracy                           0.55      9283
   macro avg       0.28      0.50      0.36      9283
weighted avg       0.30      0.55      0.39      9283



Questi dati sono in linea con la distribuzione della classe di maggioranza. I modelli Più avanzati dovranno battere il **55% di accuratezza**.

## 3.4 - Modello Naive: Momentum Persistance

Valutiamo ora la persistenza del trend: predice che Euribor_3M aumenterà se è già in aumento negli ultimi 30 giorni. 

In [86]:
# Assumiamo di usare la stessa classe del lag a 63 giorni come previsione (90 giorni di calendario = 63 giorni lavorativi)
# Scommetto 1 se il trend passato era in salita, 0 altrimenti

trend_lag_63 = df_macro_lagged['Euribor_3M'] > df_macro_lagged['Euribor_3M_lag_63d']
y_naive_trend = trend_lag_63.astype(int)

print("--- Baseline 2: Lagged Trend ---")
print(f"\nClassification Report:\n{classification_report(y_true, y_naive_trend, zero_division=0)}")
print(f"ROC-AUC: {roc_auc_score(y_true, y_naive_trend):.3f}")

--- Baseline 2: Lagged Trend ---

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.77      0.78      5122
           1       0.73      0.74      0.73      4161

    accuracy                           0.76      9283
   macro avg       0.75      0.75      0.75      9283
weighted avg       0.76      0.76      0.76      9283

ROC-AUC: 0.755


Abbiamo un'accuratezza del **76%** questo perchè la BCE segue il suo trend per anni quindi necessariamente è molto probabile che se sono saliti 90 giorni fà contineranno a farlo. Questo fenomeno si chiama **Forte Autocorrelazione** (Inerzia). Il modello "Trend" intercetta questa inerzia alla perfezione.

## 3.5 - Modello Random Forest

In [87]:
# Escludiamo dal dataset le colonne non utili per la modellazione (target, date ecc)
columns_to_exclude = ['Euribor_3M_target', 'target']
feature_columns = [col for col in df_macro_lagged.columns if col not in columns_to_exclude]

# Usiamo .to_numpy(dtype=int) per forzare il tipo di dato per evitare il warning sui dati non correttamente convertiti
y = df_macro_lagged['target'].to_numpy(dtype=int)
X = df_macro_lagged[feature_columns].to_numpy(dtype=float) # Stessa cosa per X per sicurezza

Usiamo la funzione gap di TimeSeriesSplit che ci permette di lasciare un gap tra training set e test set, in modo da evitare **Overlap Leakage**

In [88]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5, gap=FORECAST_HORIZON)

In [89]:
# Prepariamo una lista per raccogliere le metriche di ogni fold

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

roc_auc_scores = []
f1_scores = []

# inizializziamo il modello (pochi alberi e max_depth bassa per evitare overfitting all'inizio)
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)

for fold, (train_index, test_index) in enumerate(tscv.split(X)):
    # 1. SPLIT: Diviamo X e y usando gli indici generati da tscv
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # 2. SCALING: Applichiamo lo standard scaler (fit solo sui dati di train)
    scaler = StandardScaler()
    
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # 3. TRAINING: Addestriamo il modello Random Forest sui dati di train scalati
    rf_model.fit(X_train_scaled, y_train)
    
    # 4. PREVISIONE 
    # predict() restiuisce 0 o 1 per l'F1-score
    y_pred = rf_model.predict(X_test_scaled)
    # predict_proba() restituisce le probabilità per ogni classe, usiamo la colonna della classe positiva (1) per il ROC-AUC
    y_proba = rf_model.predict_proba(X_test_scaled)[:, 1]

    # 5. VALUTAZIONE: Calcoliamo F1-score e ROC-AUC per questo fold
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)
    f1_scores.append(f1)
    roc_auc_scores.append(roc_auc)

    print(f"Fold {fold+1}: Train size={len(X_train)}, Test size={len(X_test)}, F1-Score={f1:.3f}, ROC-AUC={roc_auc:.3f}")

print(f"\nRisultato Finale Random Forest (Media su {tscv.n_splits} folds):")
print(f"Mean ROC-AUC: {np.mean(roc_auc_scores):.3f}")
print(f"Mean F1-Score: {np.mean(f1_scores):.3f}")

Fold 1: Train size=1458, Test size=1547, F1-Score=0.502, ROC-AUC=0.490
Fold 2: Train size=3005, Test size=1547, F1-Score=0.630, ROC-AUC=0.826
Fold 3: Train size=4552, Test size=1547, F1-Score=0.366, ROC-AUC=0.368
Fold 4: Train size=6099, Test size=1547, F1-Score=0.615, ROC-AUC=0.433
Fold 5: Train size=7646, Test size=1547, F1-Score=0.639, ROC-AUC=0.619

Risultato Finale Random Forest (Media su 5 folds):
Mean ROC-AUC: 0.547
Mean F1-Score: 0.550


## 3.6 - Modello LSTM (Long Short-Term Memory)

Importiamo il dataset e come per gli altri modelli creaiamo le colonne target 

In [90]:
# Importiamo il dataset macro senza lag
df_macro_pp = pd.read_csv('./data/df_macro_long.csv', index_col=0 )

FORECAST_HORIZON = 90

# Creiamo la colonna col valore futuro
df_macro_pp['Euribor_3M_target'] = df_macro_pp['Euribor_3M'].shift(-FORECAST_HORIZON)

# Droppiamo gli ultimi 90 giorni (che ora sono NaN) PRIMA di fare il test
df_macro_pp = df_macro_pp.dropna(subset=['Euribor_3M_target']).copy()

# Creiamo il target binario in totale sicurezza
df_macro_pp['target'] = (df_macro_pp['Euribor_3M_target'] > df_macro_pp['Euribor_3M']).astype(int)

# Isoliamo X e y
feature_cols = ['ECB_Deposit_Rate', 'ECB_MRO_Rate', 'FED_Rate', 'VIX', 'ESI_Index',
                'Euribor_3M', 'Euribor_1Y', 'HICP_Euroarea', 'interbank_stress_spread',
                'short_yield_curve_slope', 'bce_fed_spread']

X_base = df_macro_pp[feature_cols].values
y_base = df_macro_pp['target'].values

print(f"Shape di X: {X_base.shape}")
print(f"Shape di y: {y_base.shape}")


Shape di X: (9535, 11)
Shape di y: (9535,)


Ci occupiamo della funzione che crea la sliding window

In [91]:

# creiamo il tensore 3D per LSTM [samples, seq_length, features]
def create_sequences(X_data, y_data, seq_length):
    """
    Trasforma array 2D in array 3D [samples, seq_length, features].
    """
    xs, ys = [], []
    for i in range(len(X_data) - seq_length + 1): # Aggiunto +1 per non perdere l'ultima riga
        # Prende la finestra di giorni (es. da 0 a 59)
        xs.append(X_data[i : (i + seq_length)])
        
        # Il target DEVE essere quello associato all'ULTIMO giorno della finestra
        # L'ultimo giorno della finestra è (i + seq_length - 1)
        ys.append(y_data[i + seq_length - 1]) 
        
    return np.array(xs), np.array(ys)

SEQ_LENGTH = 180 # Quanti giorni nel passato LSTM deve guardare per fare la previsione


Creiamo la struttura della rete neurale

In [92]:
# Creiamo la classe LSTM
class MacroLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout_rate):
        super(MacroLSTM, self).__init__()
        
        # layers LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout_rate) #  batch_first=True è FONDAMENTALE perché i nostri dati avranno forma (batch_size, seq_length, features)
        
        self.dropout = nn.Dropout(dropout_rate) # dropout per regolarizzazione

        # layer finale di classificazione
        self.fc = nn.Linear(hidden_size, 1)  # output binario
        self.sigmoid = nn.Sigmoid()  # per convertire l'output in probabilità

    def forward(self, x):
        # x shape: (batch, seq_len, features)
        lstm_out, (hn, cn) = self.lstm(x)  # lstm_out shape: (batch, seq_len, hidden_size)

        # Prendiamo solo l'output dell'ultimo timestep per la classificazione
        last_time_step_out = lstm_out[:, -1, :]  # shape: (batch, hidden_size)

        out = self.dropout(last_time_step_out)  # applichiamo dropout
        out = self.fc(out)  # shape: (batch, 1)
        return self.sigmoid(out)  # shape: (batch, 1) con valori tra 0 e 1

Prepariamo i dati per la k-fold assicurandoci di scalare i dati prima di creare le finestre 3d, ma dopo aver fatto lo split temporale altrimenti abbiamo data leakage

In [ ]:
# Seleziono le colonne da usare come features
feature_base_cols = [col for col in df_macro_pp.columns if col != 'target' and col != 'Euribor_3M_target']

# Prepariamo la base
X_base = df_macro_pp[feature_base_cols].values
y_base = df_macro_pp['target'].values

tscv = TimeSeriesSplit(n_splits=5, gap=FORECAST_HORIZON) # Il gap è vitale!

# Liste per salvare i risultati
lstm_roc_aucs = []
lstm_f1_scores = []

for fold, (train_index, test_index) in enumerate(tscv.split(X_base)):
    print(f"\n--- FOLD {fold+1} ---")
    
    # Split 2D
    X_train_2d, X_test_2d = X_base[train_index], X_base[test_index]
    y_train_2d, y_test_2d = y_base[train_index], y_base[test_index]
    
    # Scaler (Fit SOLO sul train)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_2d)
    X_test_scaled = scaler.transform(X_test_2d)
    
    # Trasformazione in 3D (Creazione sequenze)
    X_train_3d, y_train_seq = create_sequences(X_train_scaled, y_train_2d, SEQ_LENGTH)
    X_test_3d, y_test_seq = create_sequences(X_test_scaled, y_test_2d, SEQ_LENGTH)
    
    # Conversione in Tensori PyTorch
    X_train_t = torch.tensor(X_train_3d, dtype=torch.float32)
    y_train_t = torch.tensor(y_train_seq, dtype=torch.float32).unsqueeze(1) # shape (N, 1)
    
    X_test_t = torch.tensor(X_test_3d, dtype=torch.float32)
    y_test_t = torch.tensor(y_test_seq, dtype=torch.float32).unsqueeze(1)
    
    # DataLoaders (per addestrare a "pacchetti" e non saturare la RAM)
    train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)
    test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=64, shuffle=False)

    # Inizializziamo il modello per QUESTO fold
    input_dim = X_train_3d.shape[2]
    model = MacroLSTM(input_size=input_dim, hidden_size=64, num_layers=2, dropout_rate=0.3)
    
    # Loss per classificazione binaria e ottimizzatore Adam
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    epochs = 20 

    # --- TRAINING ---
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
    # --- EVALUATION SUL TEST SET DEL FOLD ---
    model.eval()
    y_preds_prob = []
    y_trues = []
    
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            probs = model(batch_X)
            y_preds_prob.extend(probs.numpy().flatten())
            y_trues.extend(batch_y.numpy().flatten())
            
    # Converti probabilità in 0 o 1 usando 0.5 come soglia
    y_preds_class = [1 if p > 0.5 else 0 for p in y_preds_prob]
    
    # Calcola e salva le metriche (da implementare con sklearn.metrics)
    roc = roc_auc_score(y_trues, y_preds_prob)
    f1 = f1_score(y_trues, y_preds_class)
    print(f"Fold ROC-AUC: {roc:.3f} | F1: {f1:.3f}")


--- FOLD 1 ---
Fold ROC-AUC: 0.351 | F1: 0.392

--- FOLD 2 ---
Fold ROC-AUC: 0.339 | F1: 0.612

--- FOLD 3 ---
Fold ROC-AUC: 0.137 | F1: 0.208

--- FOLD 4 ---
Fold ROC-AUC: 0.353 | F1: 0.000

--- FOLD 5 ---
